# CompanyOS — Training Script
**Environment:** CompanyOS on HuggingFace Spaces
**Method:** GRPO via HuggingFace TRL + Unsloth
**Goal:** Agent learns to complete enterprise workflows across TicketDesk, DataHub, ApprovalFlow


In [ ]:
# ── Install deps ──────────────────────────────────────────────────────────
!pip install -q unsloth trl transformers datasets wandb matplotlib requests


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
ENV_URL        = "https://YOUR-SPACE.hf.space"   # <-- set your HF Space URL
MODEL_NAME     = "unsloth/Qwen2.5-1.5B-Instruct" # small, fast for demo
MAX_EPISODES   = 200
MAX_STEPS      = 20
LOG_EVERY      = 10
WANDB_PROJECT  = "companyos-rl"                  # optional


In [ ]:
# ── Env client (calls HF Spaces REST API) ─────────────────────────────────
import requests, json

def env_reset(task_id=None):
    body = {"task_id": task_id} if task_id else {}
    r = requests.post(f"{ENV_URL}/reset", json=body, timeout=30)
    return r.json()["observation"]

def env_step(app, method, params=None):
    body = {"app": app, "method": method, "params": params or {}}
    r = requests.post(f"{ENV_URL}/step", json=body, timeout=30)
    d = r.json()
    return d["observation"], d["reward"], d["done"], d["info"]

# Sanity check
obs = env_reset()
print("Task:", obs["task"])
print("Tools:", list(obs["tools"].keys()))


In [ ]:
# ── Load model with Unsloth ────────────────────────────────────────────────
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "v_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
)
print("Model loaded.")


In [ ]:
# ── Agent: obs → action via LLM ───────────────────────────────────────────
import re

SYSTEM_PROMPT = """You are an enterprise workflow agent.
You must complete the given task by calling tools across three apps:
  - ticketdesk: list_tickets, search_tickets, get_ticket, update_ticket
  - datahub: list_metrics, query_metric, refresh_data, get_approver
  - approvalflow: list_approval_types, submit_approval, check_status, escalate

Always respond with a JSON action in this exact format:
{"app": "<app>", "method": "<method>", "params": {<params>}}
No explanation. Only valid JSON."""

def obs_to_prompt(obs):
    return (
        f"Task: {obs['task']}\n"
        f"Step: {obs['step']}/{obs['max_steps']}\n"
        f"Progress: {obs['progress']}\n"
        f"Last result: {json.dumps(obs['last_result'])}\n"
        f"\nWhat is your next action?"
    )

def agent_act(obs):
    """Sample an action from the model given the current observation."""
    prompt = obs_to_prompt(obs)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            inputs, max_new_tokens=128,
            temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)

    # Parse JSON action from model output
    try:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        action = json.loads(match.group()) if match else {}
    except Exception:
        action = {}

    # Fallback to safe default if parsing fails
    if not action.get("app") or not action.get("method"):
        action = {"app": "ticketdesk", "method": "list_tickets", "params": {}}

    return action


In [ ]:
# ── GRPO Training Loop with TRL ────────────────────────────────────────────
# We collect (prompt, completion, reward) tuples per episode,
# then run a GRPO update after each episode batch.
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
import matplotlib.pyplot as plt

episode_rewards   = []
episode_successes = []
all_prompts       = []
all_completions   = []
all_rewards_flat  = []

print("Starting training...")

for episode in range(MAX_EPISODES):
    obs       = env_reset()
    done      = False
    ep_reward = 0.0
    ep_prompts, ep_completions, ep_rewards = [], [], []

    while not done:
        prompt = obs_to_prompt(obs)
        action = agent_act(obs)
        completion = json.dumps(action)

        obs, reward, done, info = env_step(
            action.get("app", ""),
            action.get("method", ""),
            action.get("params", {})
        )

        ep_prompts.append(prompt)
        ep_completions.append(completion)
        ep_rewards.append(reward)
        ep_reward += reward

    episode_rewards.append(ep_reward)
    episode_successes.append(info.get("success", False))
    all_prompts.extend(ep_prompts)
    all_completions.extend(ep_completions)
    all_rewards_flat.extend(ep_rewards)

    # GRPO update every 10 episodes
    if (episode + 1) % 10 == 0:
        ds = Dataset.from_dict({
            "prompt":     all_prompts[-200:],
            "completion": all_completions[-200:],
            "reward":     all_rewards_flat[-200:],
        })

        grpo_config = GRPOConfig(
            output_dir="/tmp/companyos_grpo",
            num_train_epochs=1,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=4,
            learning_rate=5e-6,
            logging_steps=1,
            report_to="none",
        )
        trainer = GRPOTrainer(
            model=model,
            tokenizer=tokenizer,
            config=grpo_config,
            train_dataset=ds,
        )
        trainer.train()

        recent_mean   = sum(episode_rewards[-10:]) / 10
        success_rate  = sum(episode_successes[-10:]) / 10
        print(f"Ep {episode+1:>4} | avg_reward={recent_mean:+.2f} | success_rate={success_rate:.0%}")


In [ ]:
# ── Plot reward curves ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

window = 10
smoothed = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(episode_rewards, alpha=0.3, color='steelblue', label='Raw')
ax1.plot(range(window-1, len(episode_rewards)), smoothed, color='steelblue', linewidth=2, label='Smoothed')
ax1.set_title('Episode Reward')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Total Reward')
ax1.legend()
ax1.grid(True, alpha=0.3)

success_smooth = np.convolve(
    [1 if s else 0 for s in episode_successes],
    np.ones(window)/window, mode='valid'
)
ax2.plot(success_smooth, color='seagreen', linewidth=2)
ax2.set_title('Task Success Rate (rolling 10)')
ax2.set_xlabel('Episode')
ax2.set_ylabel('Success Rate')
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150)
plt.show()
print("Saved reward_curve.png")


In [ ]:
# ── Save model to HuggingFace Hub ──────────────────────────────────────────
# model.save_pretrained("companyos-agent")
# tokenizer.save_pretrained("companyos-agent")
# model.push_to_hub("your-hf-username/companyos-agent")
print("Training complete!")
